# Ejercicio 1: embeddings y similitud semántica

Corré las celdas en orden, de arriba hacia abajo (Shift+Enter).

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

/Users/ivanfernandez/Documents/sony/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1024 19:17:20.496875 28957 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 1. Datos
La ruta empieza con `../` porque el notebook se ejecuta desde la carpeta `experiments/`.

In [2]:
IDS = [
    "tt14205554",  # K-Pop Demon Hunters
    "tt12042730",  # Project Hail Mary
    "tt1136617",   # The Killer (2023, Fincher)
    "tt22022452",  # Inside Out 2
    "tt26342662",  # M3GAN 2.0
    "tt12747748",  # Leave the World Behind
]
df = pd.read_csv("../data/dataset_A.csv")
movies = df[df["TITLE_ID"].isin(IDS)].reset_index(drop=True)
movies[["TITLE_ID", "ORIGINAL_TITLE", "GENRES", "PLOT_SUMMARY"]]

,TITLE_ID,ORIGINAL_TITLE,GENRES,PLOT_SUMMARY
0,tt12042730,Project Hail Mary,"Adventure, Comedy, Drama, Sci-Fi",Middle school science teacher Ryland Grace (Ry...
1,tt14205554,K-Pop Demon Hunters,"Animation, Action, Adventure, Comedy, Family, ...",A world-renowned K-pop girl group balances the...
2,tt26342662,M3GAN 2.0,"Action, Horror, Sci-Fi, Thriller","Two years after the events of the first film, ..."
3,tt22022452,Inside Out 2,"Animation, Adventure, Comedy, Drama, Family, F...",Teenager Riley's mind headquarters is undergoi...
4,tt12747748,Leave the World Behind,"Drama, Mystery, Thriller",Bearing grim news of an unknown enemy and the ...
5,tt1136617,The Killer,"Action, Adventure, Crime, Drama, Thriller","A man solitary and cold, methodical and unencu..."


## 2. Modelo
Esta celda tarda unos segundos. Una vez cargado, el modelo queda en memoria: no hace falta volver a correrla.

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8127.43it/s]


## 3. Embeddings de las películas

In [4]:
movie_vectors = model.encode(movies["PLOT_SUMMARY"].tolist(), normalize_embeddings=True)
print("Forma:", movie_vectors.shape)
print("Primeros 5 números:", movie_vectors[0][:5].round(3))
print("Longitud:", np.linalg.norm(movie_vectors[0]).round(3))

Forma: (6, 384)
Primeros 5 números: [-0.039  0.013  0.011  0.021  0.067]
Longitud: 1.0


## 4. Búsqueda
Definimos una función para poder probar consultas nuevas con una sola línea.

In [5]:
def search(query):
    q_vector = model.encode(query, normalize_embeddings=True)
    scores = movie_vectors @ q_vector
    return (
        pd.DataFrame({"title": movies["ORIGINAL_TITLE"], "score": scores})
        .sort_values("score", ascending=False)
        .reset_index(drop=True)
    )

In [6]:
search("a lonely hitman losing his nerve")

,title,score
0,The Killer,0.508987
1,Project Hail Mary,0.210805
2,Inside Out 2,0.179759
3,Leave the World Behind,0.177362
4,M3GAN 2.0,0.060349
5,K-Pop Demon Hunters,-0.024782


In [7]:
search("family movies about growing up and emotions")

,title,score
0,Inside Out 2,0.287102
1,Leave the World Behind,0.187838
2,Project Hail Mary,0.171675
3,K-Pop Demon Hunters,0.143347
4,M3GAN 2.0,0.141012
5,The Killer,0.070302


In [8]:
search("movies about artificial intelligence")

,title,score
0,M3GAN 2.0,0.437463
1,Project Hail Mary,0.184744
2,The Killer,0.163469
3,Leave the World Behind,0.144752
4,Inside Out 2,0.141084
5,K-Pop Demon Hunters,0.126036


In [9]:
search("a mysterious catastrophe during a vacation")

,title,score
0,Leave the World Behind,0.505341
1,Project Hail Mary,0.353875
2,The Killer,0.296286
3,Inside Out 2,0.247224
4,M3GAN 2.0,0.125984
5,K-Pop Demon Hunters,0.062091


## Ejercicios

1. Antes de correr cada búsqueda, anotá qué película esperás primero.
2. Probá dos consultas que no compartan palabras con ninguna sinopsis.
3. Embebé título + géneros + sinopsis en lugar de solo la sinopsis y compará los rankings.
4. Quitá `normalize_embeddings=True` y mirá qué longitud se imprime.

In [10]:
# Ejercicio 2: tus consultas
search("a movie that happened in the space")

,title,score
0,Project Hail Mary,0.438671
1,M3GAN 2.0,0.230594
2,Leave the World Behind,0.202881
3,Inside Out 2,0.171090
4,The Killer,0.141672
5,K-Pop Demon Hunters,0.070966


In [ ]:
# Ejercicio 3: otro texto para embeber


In [ ]:
# Ejercicio 4: sin normalizar


In [14]:
plots = movies[["PLOT_SUMMARY"]]
for p in plots["PLOT_SUMMARY"]:
    print(p)
    print(len(p))
    




Middle school science teacher Ryland Grace (Ryan Gosling) wakes up on a spaceship light-years from home with no recollection of who he is or how he got there. As his memory returns, he begins to uncover his mission: to solve the riddle of the mysterious substance that is causing the sun to die out. He must call on his scientific knowledge and unorthodox ideas to save everything on Earth from extinction... but an unexpected friendship means he may not have to do it alone.
475
A world-renowned K-pop girl group balances their lives in the spotlight with their secret identities as bad-ass demon hunters, set against a colorful backdrop of fashion, food, style, and the most popular music movement of the current generation. When they aren't selling out stadiums, K-pop superstars Rumi, Mira and Zoey use their secret identities as bad-ass demon hunters to protect their fans from an ever-present supernatural threat. Together, they must face their biggest enemy yet - an irresistible rival boy ban